In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
!pip install -U transformers datasets accelerate

In [ ]:
data = """
Title: The Young Programmer

A young programmer wanted to build useful applications. Every evening, he practiced programming and learned something new. At first, his programs had many errors, but he continued learning. After several weeks, he successfully created his first application.

Title: The Power of Practice

A student wanted to improve his technical skills. He practiced every day and carefully studied his mistakes. Progress was slow in the beginning, but consistent practice helped him become more confident.

Title: The Curious Engineer

A curious engineering student enjoyed understanding how technology worked. Whenever he faced a difficult problem, he researched the topic and tried different solutions. His curiosity helped him learn new concepts.

Title: Learning From Failure

A developer created an application that did not work correctly. Instead of giving up, he examined the errors and fixed them one by one. The experience taught him that mistakes can become valuable learning opportunities.

Title: A New Idea

An engineering student had an idea for a useful application. He started with a simple design and gradually improved it. After testing several versions, he finally created a working prototype.

Title: The Daily Goal

A student decided to learn one new programming concept every day. Some concepts were difficult, but he remained consistent. Over time, the small daily improvements became significant progress.

Title: The Helpful Developer

A developer noticed that his classmates were struggling with a repetitive task. He created a small program to make the task easier. His simple solution saved everyone time and encouraged him to build more useful tools.

Title: Building Confidence

A beginner was nervous about writing his first program. He started with simple examples and slowly moved toward more difficult projects. With every successful program, his confidence increased.

Title: The Problem Solver

An engineer faced a complex technical problem during a project. He divided the problem into smaller parts and solved each part carefully. Eventually, the complete solution became much easier to understand.

Title: The Future Engineer

A student dreamed of becoming a successful engineer. He knew that achieving his goal would require patience, learning, and hard work. Instead of waiting for success, he focused on improving himself every day.
"""

with open("custom_dataset.txt", "w", encoding="utf-8") as file:
    file.write(data)

print("Dataset created successfully!")

Dataset created successfully!


In [ ]:
# Load and check our custom dataset

with open("custom_dataset.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Dataset loaded successfully!")
print("Number of characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])

Dataset loaded successfully!
Number of characters: 2391

First 500 characters:


Title: The Young Programmer

A young programmer wanted to build useful applications. Every evening, he practiced programming and learned something new. At first, his programs had many errors, but he continued learning. After several weeks, he successfully created his first application.

Title: The Power of Practice

A student wanted to improve his technical skills. He practiced every day and carefully studied his mistakes. Progress was slow in the beginning, but consistent practice helped him b


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Load pretrained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("GPT-2 tokenizer and model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 tokenizer and model loaded successfully!


In [ ]:
# Tokenize our custom dataset

tokenizer.pad_token = tokenizer.eos_token

encodings = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

print("Dataset tokenized successfully!")
print("Number of tokens:", encodings["input_ids"].shape[1])

Dataset tokenized successfully!
Number of tokens: 458


In [ ]:
from datasets import Dataset

# Create a dataset from our tokenized text
train_dataset = Dataset.from_dict({
    "input_ids": encodings["input_ids"].tolist(),
    "attention_mask": encodings["attention_mask"].tolist()
})

# GPT-2 should use the same tokens as labels for causal language modeling
train_dataset = train_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

# Set padding token
model.config.pad_token_id = tokenizer.pad_token_id

print("Training dataset prepared successfully!")
print("Number of training examples:", len(train_dataset))
print("Number of tokens:", len(train_dataset[0]["input_ids"]))

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Training dataset prepared successfully!
Number of training examples: 1
Number of tokens: 458


In [ ]:
# Create a larger custom dataset for GPT-2 fine-tuning

stories = [
"""Title: The Young Programmer
A young programmer wanted to build useful applications. Every evening, he practiced programming and learned something new. At first, his programs had many errors, but he continued learning. After several weeks, he successfully created his first application.""",

"""Title: The Power of Practice
A student wanted to improve his technical skills. He practiced every day and carefully studied his mistakes. Progress was slow in the beginning, but consistent practice helped him become more confident and skilled.""",

"""Title: The Curious Engineer
A curious engineering student enjoyed understanding how technology worked. Whenever he faced a difficult problem, he researched the topic and tried different solutions. His curiosity helped him learn new concepts.""",

"""Title: Learning From Failure
A developer created an application that did not work correctly. Instead of giving up, he examined the errors and fixed them one by one. The experience taught him that mistakes can become valuable learning opportunities.""",

"""Title: A New Idea
An engineering student had an idea for a useful application. He started with a simple design and gradually improved it. After testing several versions, he finally created a working prototype.""",

"""Title: The Daily Goal
A student decided to learn one new programming concept every day. Some concepts were difficult, but he remained consistent. Over time, the small daily improvements became significant progress.""",

"""Title: The Helpful Developer
A developer noticed that his classmates were struggling with a repetitive task. He created a small program to make the task easier. His simple solution saved everyone time and encouraged him to build more useful tools.""",

"""Title: Building Confidence
A beginner was nervous about writing his first program. He started with simple examples and slowly moved toward more difficult projects. With every successful program, his confidence increased.""",

"""Title: The Problem Solver
An engineer faced a complex technical problem during a project. He divided the problem into smaller parts and solved each part carefully. Eventually, the complete solution became easier to understand.""",

"""Title: The Future Engineer
A student dreamed of becoming a successful engineer. He knew that achieving his goal would require patience, learning, and hard work. Instead of waiting for success, he focused on improving himself every day.""",

"""Title: The First Project
A student wanted to create a software project for his portfolio. He first planned the requirements and then designed a simple solution. After writing the code, he tested the application and fixed several problems.""",

"""Title: Technology and Learning
Technology changes quickly, so developers need to keep learning. A responsible developer regularly studies new tools and concepts. Continuous learning helps developers adapt to new challenges.""",

"""Title: Solving a Difficult Problem
A programmer received a difficult programming problem. Instead of trying to solve everything at once, he broke the problem into smaller steps. This approach made the problem easier to understand and solve.""",

"""Title: Teamwork
A group of students worked together on a technical project. Each student had a different responsibility. By communicating regularly and helping each other, they completed the project successfully.""",

"""Title: The Coding Habit
A beginner started coding for thirty minutes every evening. In the beginning, writing programs felt difficult. After several weeks of regular practice, programming became easier and more enjoyable.""",

"""Title: The Software Developer
A software developer wanted to create applications that were simple and useful. He listened to user feedback and improved his designs. His focus on solving real problems helped him develop better applications.""",

"""Title: An Engineering Challenge
An engineering team faced a challenging project deadline. They created a clear plan and divided the work among team members. Careful planning helped them complete the important tasks on time.""",

"""Title: The Importance of Testing
A developer finished writing an application and started testing it. He discovered several small errors that were not visible during development. Testing helped him improve the reliability of the application.""",

"""Title: Learning Python
A student started learning Python because he wanted to understand programming. He began with variables and loops and later learned functions and data structures. Regular practice helped him understand programming concepts.""",

"""Title: The Goal
A student set a goal to become a skilled technology professional. He created a study schedule and worked consistently. Although the journey was challenging, every new skill brought him closer to his goal."""
]

data = "\n\n".join(stories)

with open("custom_dataset.txt", "w", encoding="utf-8") as file:
    file.write(data)

print("Larger custom dataset created successfully!")
print("Number of stories:", len(stories))
print("Number of characters:", len(data))

Larger custom dataset created successfully!
Number of stories: 20
Number of characters: 4707


In [ ]:
# Load the updated custom dataset

with open("custom_dataset.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Updated dataset loaded successfully!")
print("Number of characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])

Updated dataset loaded successfully!
Number of characters: 4707

First 500 characters:

Title: The Young Programmer
A young programmer wanted to build useful applications. Every evening, he practiced programming and learned something new. At first, his programs had many errors, but he continued learning. After several weeks, he successfully created his first application.

Title: The Power of Practice
A student wanted to improve his technical skills. He practiced every day and carefully studied his mistakes. Progress was slow in the beginning, but consistent practice helped him beco


In [ ]:
# Tokenize the updated custom dataset

tokenizer.pad_token = tokenizer.eos_token

encodings = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

print("Dataset tokenized successfully!")
print("Number of tokens:", encodings["input_ids"].shape[1])

Dataset tokenized successfully!
Number of tokens: 512


In [ ]:
from datasets import Dataset

# Create a dataset from our tokenized text
train_dataset = Dataset.from_dict({
    "input_ids": encodings["input_ids"].tolist(),
    "attention_mask": encodings["attention_mask"].tolist()
})

# GPT-2 uses the same tokens as labels for causal language modeling
train_dataset = train_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

# Set padding token
model.config.pad_token_id = tokenizer.pad_token_id

print("Training dataset prepared successfully!")
print("Number of training examples:", len(train_dataset))
print("Number of tokens:", len(train_dataset[0]["input_ids"]))

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Training dataset prepared successfully!
Number of training examples: 1
Number of tokens: 512


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-custom",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=50,
    logging_steps=10,
    learning_rate=5e-5,
    report_to="none"
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print("Trainer created successfully!")

Trainer created successfully!


In [ ]:
# Start training

print("Training started...")

trainer.train()

print("Training completed successfully!")

Training started...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed successfully!


In [ ]:
# Save the trained model and tokenizer

output_dir = "./gpt2-custom"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Model and tokenizer saved successfully!")
print("Saved at:", output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
Saved at: ./gpt2-custom


In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load our fine-tuned model
model_path = "./gpt2-custom"

tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Give the model a prompt
prompt = "A young programmer"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate text
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Prompt:")
print(prompt)

print("\nGenerated Text:")
print(generated_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt:
A young programmer

Generated Text:
A young programmer from China, Yudao, is now ready to go on the offensive.

Yudao works at a mobile app company called Mango, which has become one of the hottest mobile apps on the Web. His app, which features a user interface similar to that of Tinder, boasts a simple swipe to send a message to a person via a smartphone or tablet, with no experience required.


In [ ]:
prompt = "Title: The Curious Engineer"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Prompt:")
print(prompt)

print("\nGenerated Text:")
print(generated_text)

Prompt:
Title: The Curious Engineer

Generated Text:
Title: The Curious Engineer

The Curious Engineer Level 1 - 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat

The Curious Engineer Level 100 Hat




In [ ]:
prompt = "The Curious Engineer"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Prompt:")
print(prompt)

print("\nGenerated Text:")
print(generated_text)

Prompt:
The Curious Engineer

Generated Text:
The Curious Engineer
A little bit of everything. It's a game about being creative, and learning from mistakes; it gets better as you grow into the character... but not until they become your own person in every way! For more information on how to play this classic indie RPG (and what makes an awesome hero) visit our "How To Play" section at http://www8bitRPGsolutionsappreci


In [ ]:
prompts = [
    "A student wanted to improve his technical skills.",
    "An engineering student had an idea",
    "A developer noticed that his classmates"
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print("=" * 70)
    print("PROMPT:", prompt)
    print("\nGENERATED:")
    print(generated_text)
    print()

PROMPT: A student wanted to improve his technical skills.

GENERATED:
A student wanted to improve his technical skills. He asked for a job, but he couldn't find one in the United States or Canada and felt frustrated that many companies didn
satisfy him by offering more jobs than they could offer them (and even though those offered were good enough). A better career would have allowed students with less education experience

PROMPT: An engineering student had an idea

GENERATED:
An engineering student had an idea for a new way to make your car's tires run faster. He saw the need in his hometown, and decided to build one himself from scratch using nothing but aluminum tubing that could be bent out of shape with no effort at all—and without any extra tools or hardware needed. The result is this

PROMPT: A developer noticed that his classmates

GENERATED:
A developer noticed that his classmates were being taught about the nature of music. They began to hear a song called "The Blues Brother

In [ ]:
# Compare the original GPT-2 with our fine-tuned GPT-2

from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load original GPT-2
original_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
original_model = GPT2LMHeadModel.from_pretrained("gpt2")
original_model.to(device)

prompt = "A young programmer"

# Function for text generation
def generate_text(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# Generate from original GPT-2
original_output = generate_text(
    original_model,
    original_tokenizer,
    prompt
)

# Generate from fine-tuned GPT-2
fine_tuned_output = generate_text(
    model,
    tokenizer,
    prompt
)

print("=" * 70)
print("PROMPT:", prompt)

print("\nORIGINAL GPT-2:")
print(original_output)

print("\nFINE-TUNED GPT-2:")
print(fine_tuned_output)

print("=" * 70)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PROMPT: A young programmer

ORIGINAL GPT-2:
A young programmer named Alex, who had a knack for programming but could not write code in his spare time. He was so enthusiastic that he decided to try the Java language and made it available on GitHub (http://github:com/javac-dev).
The goal of this project is an easy way

FINE-TUNED GPT-2:
A young programmer, the first to build a JavaScript app in Node.js and then go back into it later on with some additional code was able create his own React application using ES6's built-in JSX library . The resulting project can be found here , but you'll need an HTML5 browser that supports


In [ ]:
# Create multiple training chunks from the complete dataset

block_size = 128

# Tokenize the complete dataset without truncating it
all_tokens = tokenizer(
    text,
    return_tensors="pt",
    truncation=False
)["input_ids"][0]

# Create chunks
chunks = []

for i in range(0, len(all_tokens), block_size):
    chunk = all_tokens[i:i + block_size]

    # Keep only complete chunks
    if len(chunk) == block_size:
        chunks.append(chunk.tolist())

print("Total tokens:", len(all_tokens))
print("Training chunks created:", len(chunks))
print("Tokens per chunk:", block_size)

Total tokens: 872
Training chunks created: 6
Tokens per chunk: 128


In [ ]:
# Create the final training dataset from all chunks

import torch
from datasets import Dataset

input_ids = torch.tensor(chunks, dtype=torch.long)

# Create attention masks
attention_mask = torch.ones_like(input_ids)

train_dataset = Dataset.from_dict({
    "input_ids": input_ids.tolist(),
    "attention_mask": attention_mask.tolist(),
    "labels": input_ids.tolist()
})

print("Final training dataset created successfully!")
print("Number of training examples:", len(train_dataset))
print("Tokens per example:", len(train_dataset[0]["input_ids"]))

Final training dataset created successfully!
Number of training examples: 6
Tokens per example: 128


In [ ]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

# Start again from the original pretrained GPT-2
base_model = GPT2LMHeadModel.from_pretrained("gpt2")

base_model.config.pad_token_id = tokenizer.pad_token_id

training_args_final = TrainingArguments(
    output_dir="./gpt2-custom-final",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

trainer_final = Trainer(
    model=base_model,
    args=training_args_final,
    train_dataset=train_dataset
)

print("Final trainer created successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Final trainer created successfully!


In [ ]:
print("Final training started...")

trainer_final.train()

print("Final training completed successfully!")

Final training started...


Step,Training Loss
1,3.411919
2,3.078082
3,3.247113
4,2.979601
5,2.868636
6,2.944683
7,2.694571
8,2.503355
9,2.662728
10,2.436318


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final training completed successfully!


In [ ]:
final_dir = "./gpt2-custom-final"

trainer_final.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print("Final model and tokenizer saved successfully!")
print("Saved at:", final_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model and tokenizer saved successfully!
Saved at: ./gpt2-custom-final


In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ORIGINAL GPT-2
original_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
original_model = GPT2LMHeadModel.from_pretrained("gpt2")
original_model.to(device)

# Load FINAL FINE-TUNED GPT-2
final_tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-custom-final")
final_model = GPT2LMHeadModel.from_pretrained("./gpt2-custom-final")
final_model.to(device)

prompt = "A young programmer"

# -------- ORIGINAL GPT-2 --------
original_inputs = original_tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    original_output = original_model.generate(
        **original_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=original_tokenizer.eos_token_id
    )

original_text = original_tokenizer.decode(
    original_output[0],
    skip_special_tokens=True
)

# -------- FINE-TUNED GPT-2 --------
final_inputs = final_tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    final_output = final_model.generate(
        **final_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=final_tokenizer.eos_token_id
    )

final_text = final_tokenizer.decode(
    final_output[0],
    skip_special_tokens=True
)

print("=" * 70)
print("PROMPT:", prompt)

print("\nORIGINAL GPT-2:")
print(original_text)

print("\nFINE-TUNED GPT-2:")
print(final_text)

print("=" * 70)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PROMPT: A young programmer

ORIGINAL GPT-2:
A young programmer named David had the idea of putting a game that would allow him to control characters through his brain. He was intrigued by it, and wanted to see if he could make such an app for Xbox One which allowed anyone who loved games from other platforms in one place without having any special knowledge about how they

FINE-TUNED GPT-2:
A young programmer named Chris was a big fan of the programming language and knew it well. He had started learning PHP at age 13, but he didn't know much about coding until his first class on building an application server program in C# with Python 3 support for Visual Studio 2015 (see this tutorial).
After


In [ ]:
prompt = "A young programmer"

# Generate from original GPT-2
original_inputs = original_tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    original_output = original_model.generate(
        **original_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=original_tokenizer.eos_token_id
    )

original_text = original_tokenizer.decode(
    original_output[0],
    skip_special_tokens=True
)

# Generate from final fine-tuned GPT-2
final_inputs = final_tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    final_output = final_model.generate(
        **final_inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=final_tokenizer.eos_token_id
    )

final_text = final_tokenizer.decode(
    final_output[0],
    skip_special_tokens=True
)

print("=" * 70)
print("PROMPT:", prompt)

print("\nORIGINAL GPT-2:")
print(original_text)

print("\nFINE-TUNED GPT-2:")
print(final_text)

print("=" * 70)

PROMPT: A young programmer

ORIGINAL GPT-2:
A young programmer, a member of the team that created this app and has been working on it for nearly two years.
I've recently started to see how hard developers can work with their code in an extremely short amount or so (if you want more detailed info about development time) but there are still many aspects

FINE-TUNED GPT-2:
A young programmer named Alex wrote a program that would create an application on top of PHP. The following code was designed to be easily integrated into any project using the standard library, but it contained bugs and needed improvements so developers could improve their applications faster with minimal effort.
The initial implementation used some simple examples taken from


README.md

In [ ]:
readme = """
# PRODIGY_GA_01 - Text Generation with GPT-2

## Objective
Fine-tune the GPT-2 language model on a custom text dataset and generate new text based on user-provided prompts.

## Technologies Used
- Python
- Google Colab
- PyTorch
- Hugging Face Transformers
- Hugging Face Datasets
- GPT-2

## Dataset
A custom dataset containing short programming, engineering, and student-related stories was created for fine-tuning.

Total tokens used for training: 872

The dataset was divided into:
- 6 training examples
- 128 tokens per example

## Model Training
The original pretrained GPT-2 model was fine-tuned for 3 epochs.

Training configuration:
- Batch size: 1
- Learning rate: 5e-5
- Epochs: 3
- Block size: 128 tokens

## Text Generation
Prompt used for testing:

A young programmer

The fine-tuned model generated programming-related text based on the prompt.

## Model Comparison
The project compares text generated by:
1. Original GPT-2
2. Fine-tuned GPT-2

The fine-tuned model was trained on the custom programming and engineering dataset.

## Limitations
The custom dataset used in this project is relatively small. Therefore, the generated text may sometimes contain unrelated details, incomplete sentences, or mixed programming concepts.

## Conclusion
The project demonstrates the complete workflow of loading GPT-2, preparing a custom dataset, fine-tuning the model, saving the trained model, and generating text from prompts.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully!")

README.md created successfully!


requirements.txt

In [ ]:
requirements = """transformers
datasets
accelerate
torch
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [ ]:
import os

print("PROJECT FILES")
print("=" * 50)

for item in os.listdir("."):
    if item.startswith("gpt2-custom"):
        print(item + "/")
    elif item in [
        "custom_dataset.txt",
        "README.md",
        "requirements.txt",
        "model_comparison.txt"
    ]:
        print(item)

print("=" * 50)
print("Project files check completed!")

PROJECT FILES
custom_dataset.txt
gpt2-custom-final/
gpt2-custom/
requirements.txt
README.md
Project files check completed!


model_comparison.txt

In [ ]:
comparison_text = f"""
GPT-2 TEXT GENERATION COMPARISON
================================

Prompt:
{prompt}

ORIGINAL GPT-2:
{original_text}

FINE-TUNED GPT-2:
{final_text}

Observation:
The fine-tuned GPT-2 generated programming and software-development related text
after being trained on the custom dataset.

The dataset was relatively small, so some generated content may be incomplete
or contain mixed programming concepts.
"""

with open("model_comparison.txt", "w", encoding="utf-8") as file:
    file.write(comparison_text)

print("model_comparison.txt created successfully!")

model_comparison.txt created successfully!


In [ ]:
import os

required_files = [
    "custom_dataset.txt",
    "README.md",
    "requirements.txt",
    "model_comparison.txt"
]

print("FINAL PROJECT CHECK")
print("=" * 50)

for file in required_files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌", file)

if os.path.exists("gpt2-custom-final"):
    print("✅ gpt2-custom-final/")

print("=" * 50)
print("Final project check completed!")

FINAL PROJECT CHECK
✅ custom_dataset.txt
✅ README.md
✅ requirements.txt
✅ model_comparison.txt
✅ gpt2-custom-final/
Final project check completed!
